# Problem Set 10

[PSet 10](pset10.pdf)


In [1]:
import sympy as sp
import sympy.physics.units as spu
import sympy.physics.mechanics as spm
import sympy.physics.vector as spv
from sympy.physics.vector.printing import init_vprinting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import display, Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
init_vprinting(use_latex="mathjax")

HALF = sp.S.Half
PI = sp.pi

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> spm.ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return spm.ReferenceFrame(
        frame,
        latexs=(
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{x}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{y}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{z}}}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> spm.ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

In [3]:
def simplify_square_symbol(value, rationaliser):
    num, den = value.as_numer_denom()
    num = (num / rationaliser**2).simplify().collect([g])
    den = (den / rationaliser**2).simplify()
    return num / den

def simplify_by_symbol(value, symbol):
    num, den = value.as_numer_denom()
    num = (num / symbol).simplify().collect([g])
    den = (den / symbol).simplify()
    return num / den

## Problem 1 Moment of Inertia: Disc and Washer

(a) A thin uniform disc of mass $M$ and radius $R$ is mounted on an
axis passing through the center of the disk, perpendicular to the plane 
of the disc. Determine the moment of inertia about the following two 
parallel axes that are perpendicular
to the plane of the disk.

(i) An axis passing through the center of mass of the disc.

(ii) An axis passing through a point on the rim of the disc a distance 
$R$ from the center.

Express your answers in some or all of the following variables: $M$ and $R$.

In [4]:
display(Markdown("![](../figures/PS10-01-Inertia-Disk.jpg)"))

R, M = sp.symbols("R M", positive=True, real=True)
lam = M / (sp.pi * R**2)

display(
    Math(
        r"\text{density of the disc (mass per unit area) }"
        r"\lambda = \frac{M}{\pi R^2}"
    )
)
r = sp.symbols("r")
moi = (2*sp.pi*r*lam*r**2).integrate((r, 0, R))
display(
    Math(
        r"\text{i. Moment of inertia of the disc about its central axis}"
        r"\text{ (perpendicular to the plane of the disc):}\\"
        rf"\boxed{{I = {sp.latex(moi)}}}"
    )
)

I_rim = moi + M*R**2
display(
    Math(
        r"\text{ii. Moment of inertia of the disc with a rim about its central axis}\\"
        rf"\boxed{{I = {sp.latex(I_rim)}}}"
    )
)

![](../figures/PS10-01-Inertia-Disk.jpg)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

(b) A 1 inch US Standard Washer has inner radius $r_i = 13.5 \mathrm{mm}$ and an outer radius
$r_o = 31.0 \mathrm{mm}$. The washer is approximately $d = 4.0 \mathrm{mm}$ thick. The density of
the washer is $\rho = 7.8 \times 10^3 \mathrm{kg\, m^{-3}}$. Calculate the moment of inertia of the washer
about an axis that is perpendicular to the plane of the washer and passes through
its center of mass.

![](../figures/PS10-01-Inertia-Disk_b.jpg)


In [5]:
(
    ri,  # inner radius of the washer
    ro,  # outer radius of the washer
    d,   # thickness of the washer
    rho  # density of the washer
) = sp.symbols("r_i r_o d rho", positive=True, real=True)

values = {
    ri: sp.Float(13.5e-3),  # inner radius in meters
    ro: sp.Float(31.0e-3),  # outer radius in meters
    d: sp.Float(4.0e-3),    # thickness in meters
    rho: sp.Float(7.8e3)    # density in kg/m^3
}


In [6]:
display(
    Markdown(r"![](../figures/PS10-01-Inertia-Disk_b1.jpg)")
)

r = sp.symbols("r", positive=True, real=True)
I_washer = sp.integrate(2*sp.pi*r*d*rho*r**2, (r, ri, ro))
display(
    Math(
        fr"I_\text{{washer}} = {sp.latex(I_washer)}"
    )
)

I_washer_value = float(I_washer.subs(values).evalf())
display(Math(rf"I_{{washer}} = {I_washer_value:.2e}\ \mathrm{{kg\,m^2}}"))

![](../figures/PS10-01-Inertia-Disk_b1.jpg)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Problem 2

![Cylinder rotating](../figures/PS10-02-Cylinder-Rotating.jpg)

Consider a fixed wheel that consists of a uniform cylindrical disk with a 
radius $R$ and a mass $M_1$, and a second uniform cylindrical disk of 
radius $R/2$ and mass $M_2$ that is attached concentrically to the first disk.

The wheel is free to rotate about a frictionless pivot at its center. 
A rope is wound around the smaller disk, then around a small frictionless post. 
The other end of the rope is attached to a block of mass $M_b$ so that 
both disks rotate together as the block falls.

The downward acceleration of gravity is $g$. The block is released from rest.

How long does it take the block to fall a distance $d$? 

Express your answer in terms of $R$, $M_1$, $M_2$, $M_b$, $g$, and $d$ as needed.

In [7]:
(
    R, # radius of the larger disk
    M1, # mass of the larger disk
    M2, # mass of the smaller disk
    Mb, # mass of the block
    g, # acceleration due to gravity
    d, # distance the block falls
    L, # length of the rope
    I # moment of inertia of the wheels
) = sp.symbols('R M1 M2 M_b g d L I')

values = {
    I: HALF*((M1*R**2 + M2*(R/2)**2).simplify()), # moment of inertia of the wheels
}

In [8]:
display(
    Markdown(
        "| FBD | Coordinate system |\n"
        "|:---:|:---:|\n"
        "| ![FBD](../figures/PS10-02-Cylinder-Rotating_1.jpg) "
        "| ![Coordinate system](../figures/PS10-02-Cylinder-Rotating_xy.jpg) |\n"
        "| ![FBD](../figures/PS10-02-Cylinder-Rotating_2.jpg) "
        "| ![Coordinate system](../figures/PS10-02-Cylinder-Rotating_xy.jpg) |"
    )
)

N = reference_frame('N')
t = spv.dynamicsymbols._t
T = sp.symbols('T', real=True, positive=True) # Tension in the string
a = sp.symbols('a', real=True, positive=True) # Linear acceleration of the cylinder
theta = spv.dynamicsymbols('theta')

tau = (R/2*(-N.y).cross(T*N.x)).dot(N.z)
N2LReqn1 = sp.Eq(tau, I*2*a/R)
display(Math(rf"\text{{Newton's 2nd Law (Rotational): }} {sp.latex(N2LReqn1)}"))

forces_on_Mb = Mb*g*N.y + T*(-N.y)
display(Math(rf"\text{{Forces on }} M_b: {sp.latex(forces_on_Mb)}"))
N2Leqn = sp.Eq(forces_on_Mb.dot(N.y), Mb*a)
display(Math(rf"\text{{Newton's 2nd Law: }} {sp.latex(N2Leqn)}"))
T_sol = sp.solve(N2Leqn, T, dict=True)[0]
display(Math(rf"\text{{Tension: }} {sp.latex(T_sol)}"))

N2LReqn2 = N2LReqn1.subs(T_sol)
display(Math(rf"\text{{Substituting T: }} {sp.latex(N2LReqn2)}"))
a_sol = sp.solve(N2LReqn2, a, dict=True)[0]
display(Math(rf"\text{{Acceleration: }} {sp.latex(a_sol)}"))

a_complete_with_MOI = simplify_square_symbol(a_sol[a].subs(I, values[I]), R)
a_complete_with_MOI = simplify_by_symbol(a_complete_with_MOI, sp.Integer(2))
display(Math(rf"\text{{Acceleration with MOI: }} \boxed{{{sp.latex(a_complete_with_MOI)}}}"))

time_to_fall_by_d = sp.sqrt(2*d/a_complete_with_MOI)
display(Math(rf"\text{{Time to fall by }} d: \boxed{{{sp.latex(time_to_fall_by_d)}}}")) 

| FBD | Coordinate system |
|:---:|:---:|
| ![FBD](../figures/PS10-02-Cylinder-Rotating_1.jpg) | ![Coordinate system](../figures/PS10-02-Cylinder-Rotating_xy.jpg) |
| ![FBD](../figures/PS10-02-Cylinder-Rotating_2.jpg) | ![Coordinate system](../figures/PS10-02-Cylinder-Rotating_xy.jpg) |

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [9]:
# Use Conservation of Energy to relate kinetic and potential energy changes

KEi = 0
KEf = HALF*I*(2*a*t/R)**2 + HALF*Mb*(a*t)**2
delta_KE = KEf - KEi

PEi = 0
PEf = -Mb*g*d
delta_PE = PEf - PEi

delta_CME = delta_KE + delta_PE
display(Math(rf"\Delta KE = {sp.latex(delta_KE)}"))
display(Math(rf"\Delta PE = {sp.latex(delta_PE)}"))
display(Math(rf"\Delta CME = {sp.latex(delta_CME)} = 0"))

delta_CME = delta_CME.subs(values).subs(a, a_complete_with_MOI)
t2_sol = sp.solve(delta_CME, t**2, dict=True)[0][t**2]
display(Math(rf"\text{{Using CME }} t = \boxed{{{sp.latex(sp.sqrt(t2_sol))}}}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [10]:
# Using the solution for y(t) to find the time to reach a specific distance d
y = sp.Function('y')(t)
doty = y.diff(t)

KEi = 0
KEf = HALF*I*(2*doty/R)**2 + HALF*Mb*doty**2
delta_KE = (KEf - KEi).simplify()

PEi = 0
PEf = -Mb*g*y
delta_PE = (PEf - PEi).simplify()

delta_CME_eqn = sp.Eq(delta_KE + delta_PE, 0).simplify()
display(Math(
    rf"\begin{{aligned}}"
        rf"\Delta KE &= {spv.vlatex(delta_KE)} \\"
        rf"\Delta PE &= {spv.vlatex(delta_PE)} \\"
        rf"\Delta CME &= {spv.vlatex(delta_CME_eqn)}"
    rf"\end{{aligned}}"
))
solution = sp.dsolve(delta_CME_eqn, y, ics={y.subs(t, 0): 0})
display(Math(rf"y(t) = {spv.vlatex(solution.rhs)}"))

solution_rhs = solution.rhs.subs(values)
solution_time = sp.sqrt(sp.solve(sp.Eq(solution_rhs, d), t**2)[0])
display(
    Math(
        rf"\text{{Time to reach distance d is }}" 
        rf"\boxed{{{spv.vlatex(solution_time)} \text{{ secs}}}} "
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Problem 3 Suspended Rod

A uniform rod of Length $L = 2.0 \text{ m}$ and mass $m = 4.0 \text{ kg}$ 
is hinged to a wall at one end and suspended from the wall by a cable that 
is attached to the other end of the rod at an angle of $\beta = 30^\circ$ 
to the rod. 

Assume the cable has zero mass. There is a contact force at the pivot on the rod. 
Use $\alpha$ to represent the angle of the pivot force 
(the force acting at the wall). The magnitude and direction of this force 
is unknown. 

In this problem, you will solve for the magnitude of the tension in the 
cable and the direction and magnitude of the pivot force.

![Suspended Rod](../figures/PS10-03-Suspended-Rod.jpg)

(a) What is the magnitude of the tension in the cable?

(b) What angle does the pivot force make with the beam in degrees?

(c) What is the magnitude of the pivot force?

![FBD](../figures/PS10-03-Suspended-Rod-01.jpg)


In [11]:
(
    L, # Length of the rod
    m, # Mass of the rod
    beta, # Angle of the cable to the rod
    alpha, # Angle of the pivot force
    g, # Acceleration due to gravity
) = sp.symbols('L m beta alpha g', real=True, positive=True)

values = {
    L: 2.0,
    m: 4.0,
    beta: sp.rad(30),
    g: 9.8,
}

(
    T,  # Tension in the cable
    Fx,  # x-direction Force at the pivot
    Fy,  # y-direction Force at the pivot
) = sp.symbols("T Fx Fy", real=True, positive=True)

N = reference_frame("N")

vec_T = T * (N.x * sp.cos(PI - beta) + N.y * sp.cos(PI / 2 + beta))
vec_F = Fx * N.x + Fy * (-N.y)
vec_W = m * g * N.y
vec_L = L * N.x

tau_hinge = (vec_L / 2).cross(vec_W) + vec_L.cross(vec_T)
tau_magnitude_squared = tau_hinge.dot(tau_hinge)
tau_equilibrium_eqn = sp.Eq(tau_magnitude_squared, 0)
tau_equilibrium_solution = sp.solve(tau_equilibrium_eqn, T)[0]

display(
    Math(
        rf"\text{{Tension in the cable }}: " 
        rf"T = {sp.latex(tau_equilibrium_solution)}")
)

T_evaluated = tau_equilibrium_solution.subs(values)
display(
    Math(
        rf"\text{{Tension in the cable evaluated }}: "
        rf"T = {sp.latex(T_evaluated)} N")
)

total_force_on_rod = vec_F + vec_W + vec_T
display(
    Math(
        rf"\text{{Total force on the rod }}: " 
        rf"\vec{{F}}_{{\text{{total}}}} = {sp.latex(total_force_on_rod)}"
    )
)

total_force_on_rod_matrix = total_force_on_rod.to_matrix(N)
total_force_on_rod_eqn = sp.Eq(total_force_on_rod_matrix, sp.zeros(3, 1))

force_solution = sp.solve(total_force_on_rod_eqn, (Fx, Fy), dict=True)[0]
display(
    Math(
        rf"\text{{Force at the pivot: }} " 
        rf"Fx = {sp.latex(force_solution[Fx])}," 
        rf"Fy = {sp.latex(force_solution[Fy])}"
    )
)

Fx_evaluated = force_solution[Fx].subs(values).subs(T, T_evaluated)
Fy_evaluated = force_solution[Fy].subs(values).subs(T, T_evaluated)
display(
    Math(
        rf"\text{{Force at the pivot evaluated: }}"
        rf"Fx = {sp.latex(Fx_evaluated)}\;, "
        rf"Fy = {sp.latex(Fy_evaluated)}\;"
        rf"F = {sp.latex(sp.sqrt(Fx_evaluated**2 + Fy_evaluated**2))}\;"
    )
)

# Fy is defined along -N.y, so the pivot-force components are (Fx, -Fy).
angle_force_on_rod = sp.N(
    sp.atan2(Fy_evaluated, Fx_evaluated) * 180 / PI
)
display(
    Math(
        rf"\text{{Angle of the total force on the rod with respect to the x-axis: }} "
        rf"\alpha = {sp.latex(angle_force_on_rod)}^\circ"
    )
)

vec_F_evaluated = Fx_evaluated * N.x + Fy_evaluated * (-N.y)
display(
    Math(
        rf"\text{{Force at the pivot vector evaluated: }} "
        rf"\vec{{F}} = {sp.latex(vec_F_evaluated)}"
    )
)

angle_between_x = sp.N(vec_F_evaluated.angle_between(N.x) * 180 / PI)
display(
    Math(
        rf"\text{{Angle of the force at the pivot with respect to the x-axis: }} "
        rf"\alpha = {sp.latex(angle_between_x)}^\circ"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Problem 4 Person on Hill
